# Image Stitching via Automatic Keypoint Matching
### NTUST CI3D — Computer Vision Lab

This notebook walks through an **automatic image stitching pipeline** that combines two
images taken from different viewpoints of the same planar scene into a single panorama.

---

## Pipeline Overview

```
Image 1 (left)  +  Image 2 (right)
        │
        ▼
 [Step 1]  SIFT  — detect keypoints & compute 128-d descriptors
        │
        ▼
 [Step 2]  FLANN k-NN matching  →  Lowe's ratio test
        │
        ▼
 [Step 3]  RANSAC  →  estimate 3×3 Homography  H
        │
        ▼
 [Step 4]  warpPerspective  +  alpha blending
        │
        ▼
       Panorama
```

**Key concepts covered:** Scale-Invariant Feature Transform (SIFT), nearest-neighbor
matching, Lowe's ratio test, Random Sample Consensus (RANSAC), projective Homography,
and perspective warping.


---
## 0. Imports


We need only three libraries:

| Library | Role |
|---------|------|
| `numpy` | Array maths and matrix operations |
| `cv2` (OpenCV) | Image I/O, SIFT, FLANN, RANSAC, warp |
| `matplotlib` | Inline display inside the notebook |


In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

# Display images inline (RGB — OpenCV loads as BGR, so we convert when plotting)
def show(img, title="", ax=None, cmap=None):
    """Show a BGR or grayscale image inline via matplotlib."""
    rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB) if img.ndim == 3 else img
    if ax is None:
        plt.figure(figsize=(8, 5))
        plt.imshow(rgb, cmap=cmap)
        plt.title(title); plt.axis("off"); plt.tight_layout(); plt.show()
    else:
        ax.imshow(rgb, cmap=cmap)
        ax.set_title(title); ax.axis("off")

print("OpenCV version:", cv2.__version__)


---
## Step 0 — Synthetic Demo Images

We build a controlled planar scene (checkerboard + colored circles + text) and extract
**two partially overlapping sub-views**, each with a different mild perspective distortion,
to simulate images captured from two distinct camera positions.

### Scene layout

```
┌─────────────────────────────────┐
│  full scene  (800 × 600)        │
│  ┌───────────┐                  │
│  │   img1    │                  │
│  │ cols 0-500│                  │
│  │     ┌─────┼──────────────┐   │
│  │     │ OVL │    img2      │   │
│  └─────┼─────┘  cols 300-800│   │
│        └──────────────────────  │
└─────────────────────────────────┘
           ↑
      overlap: cols 300–500
```

Both sub-views are warped with **different** perspective transforms to mimic the
viewpoint change between two cameras.


In [ ]:
img1 = cv2.imread('demo_left.jpg')
img2 = cv2.imread('demo_right.jpg')


fig, axes = plt.subplots(1, 2, figsize=(12, 5))
show(img1, "Image 1 (left view)",  axes[0])
show(img2, "Image 2 (right view)", axes[1])
plt.suptitle("Step 0 — Input Images", fontsize=14, fontweight="bold")
plt.tight_layout(); plt.show()

print(f"Image 1 size: {img1.shape[1]}×{img1.shape[0]}")
print(f"Image 2 size: {img2.shape[1]}×{img2.shape[0]}")


---
## Step 1 — SIFT Feature Detection & Descriptor Computation

### What is SIFT?

**Scale-Invariant Feature Transform (SIFT)** detects *keypoints* — distinctive local
regions of an image — and describes each one with a **128-dimensional vector** that is
invariant to scale, rotation, and partially invariant to illumination change.

### How it works

1. **Scale-space extrema detection** — build a Gaussian pyramid (DoG) and find local
   extrema across scale and space.
2. **Keypoint localization** — sub-pixel refinement; discard low-contrast or edge-like
   responses.
3. **Orientation assignment** — assign a dominant gradient orientation; this grants
   rotation invariance.
4. **Descriptor computation** — divide the 16×16 patch around each keypoint into a 4×4
   grid of cells; compute an 8-bin gradient histogram per cell →
   4×4×8 = **128 values**.

```
Keypoint  →  [d₁, d₂, ..., d₁₂₈]   (L2-normalized float32 vector)
```


In [ ]:
def detect_and_compute(img_gray: np.ndarray):
    """
    Detect keypoints and compute SIFT descriptors on a grayscale image.

    Args:
        img_gray: Grayscale input image of shape (H, W).

    Returns:
        kps   : List of cv2.KeyPoint objects.
        descs : Descriptor array of shape (N, 128), dtype float32.
    """
    sift = cv2.SIFT_create()
    kps, descs = sift.detectAndCompute(img_gray, None)
    return kps, descs

# Convert to grayscale before detection
gray1 = cv2.cvtColor(img1, cv2.COLOR_BGR2GRAY)
gray2 = cv2.cvtColor(img2, cv2.COLOR_BGR2GRAY)

kps1, descs1 = detect_and_compute(gray1)
kps2, descs2 = detect_and_compute(gray2)

print(f"Image 1: {len(kps1):4d} keypoints  |  descriptor shape: {descs1.shape}")
print(f"Image 2: {len(kps2):4d} keypoints  |  descriptor shape: {descs2.shape}")

# Visualize keypoints (draw circles sized by scale)
vis1 = cv2.drawKeypoints(img1, kps1, None,
                          flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS)
vis2 = cv2.drawKeypoints(img2, kps2, None,
                          flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
show(vis1, f"Image 1 — {len(kps1)} SIFT keypoints", axes[0])
show(vis2, f"Image 2 — {len(kps2)} SIFT keypoints", axes[1])
plt.suptitle("Step 1 — SIFT Keypoints (circle size ∝ scale)", fontsize=14, fontweight="bold")
plt.tight_layout(); plt.show()


---
## Step 2 — FLANN Matching + Lowe's Ratio Test

### FLANN k-NN Matching

**FLANN** (Fast Library for Approximate Nearest Neighbors) uses a KD-tree index to
efficiently find, for each descriptor in image 1, its **two nearest neighbors** in
image 2's descriptor space (k=2).

### Lowe's Ratio Test

A raw k-NN match can be ambiguous — the best match might be nearly as far as the
second-best. Lowe's ratio test filters these out:

$$
\frac{d_1}{d_2} < r \quad (r = 0.75 \text{ by default})
$$

Keep the match only if the best-match distance $d_1$ is **significantly smaller**
than the second-best $d_2$. This retains *distinctive* matches and discards
ambiguous ones.

```
k-NN result:   (m₁, distance d₁)   ← best match
               (m₂, distance d₂)   ← second-best

Keep if:   d₁ / d₂  <  0.75
```


In [ ]:
def match_features(descs1: np.ndarray, descs2: np.ndarray,
                   ratio: float = 0.75):
    """
    Match descriptors using FLANN KD-tree k-NN (k=2) and filter with
    Lowe's ratio test.

    Args:
        descs1 : Descriptors from image 1.
        descs2 : Descriptors from image 2.
        ratio  : Lowe's ratio threshold (default 0.75).

    Returns:
        good : List of cv2.DMatch objects that passed the ratio test.
    """
    # algorithm=1 selects FLANN_INDEX_KDTREE; trees=5 balances speed/accuracy
    index_params  = dict(algorithm=1, trees=5)
    search_params = dict(checks=50)

    flann   = cv2.FlannBasedMatcher(index_params, search_params)
    matches = flann.knnMatch(descs1, descs2, k=2)

    good = []
    for m, n in matches:
        if m.distance < ratio * n.distance:
            good.append(m)
    return good

RATIO = 0.75
good  = match_features(descs1, descs2, ratio=RATIO)

total_knn = len(descs1)
print(f"Total k-NN pairs:          {total_knn}")
print(f"Passed ratio test (r={RATIO}): {len(good)}  ({100*len(good)/total_knn:.1f}%)")

# Draw ALL good matches (before RANSAC filtering)
vis_all = cv2.drawMatches(img1, kps1, img2, kps2, good, None,
                           matchColor=(0, 200, 255),
                           singlePointColor=(200, 200, 200),
                           flags=cv2.DrawMatchesFlags_DEFAULT)
show(vis_all, f"Step 2 — {len(good)} good matches after Lowe's ratio test (r={RATIO})")


---
## Step 3 — Homography Estimation with RANSAC

### What is a Homography?

A **Homography** $H$ is a $3 \times 3$ projective transformation matrix. For two
images of the **same planar surface** (or for a camera that only rotates), every
corresponding point pair $(\mathbf{x}_1,\, \mathbf{x}_2)$ satisfies:

$$
\tilde{\mathbf{x}}_1 \;\sim\; H\,\tilde{\mathbf{x}}_2
$$

where $\tilde{\mathbf{x}} = [x,\,y,\,1]^\top$ is the homogeneous coordinate.
$H$ has 8 degrees of freedom (9 entries, scale-invariant), so a minimum of
**4 point correspondences** are required to solve for it.

### Why RANSAC?

Even after Lowe's ratio test, some matches are still wrong (outliers). RANSAC
(**Random Sample Consensus**) robustly estimates $H$ despite outliers:

1. Randomly sample **4** point pairs.
2. Compute a candidate $H$ from those 4 pairs.
3. Count **inliers** — pairs whose reprojection error $\|\mathbf{x}_1 - H\mathbf{x}_2\| < \epsilon$.
4. Repeat many iterations; keep the $H$ with the most inliers.
5. Re-fit $H$ using all inliers of the best hypothesis.


In [ ]:
def compute_homography(kps1, kps2, good_matches,
                       reproj_thresh: float = 4.0):
    """
    Robustly estimate a Homography H from matched point pairs using RANSAC,
    such that:   x1 ≈ H · x2   (maps image-2 points into image-1 coordinates).

    Args:
        kps1          : Keypoints from image 1.
        kps2          : Keypoints from image 2.
        good_matches  : Filtered match list from match_features().
        reproj_thresh : RANSAC reprojection error threshold in pixels.

    Returns:
        H    : 3x3 Homography matrix (np.ndarray), or None on failure.
        mask : Inlier/outlier mask of shape (N, 1), dtype uint8.
    """
    if len(good_matches) < 4:
        print(f"[ERROR] Not enough good matches ({len(good_matches)} < 4).")
        return None, None

    # Extract matched point coordinates
    pts1 = np.float32([kps1[m.queryIdx].pt for m in good_matches])
    pts2 = np.float32([kps2[m.trainIdx].pt for m in good_matches])

    # findHomography(src, dst) — here src=pts2, dst=pts1  (img2 → img1)
    H, mask = cv2.findHomography(pts2, pts1, cv2.RANSAC, reproj_thresh)
    return H, mask

REPROJ_THRESH = 4.0
H, mask = compute_homography(kps1, kps2, good, reproj_thresh=REPROJ_THRESH)

n_inlier  = int(mask.sum())
n_outlier = len(good) - n_inlier
print(f"Matches:  {len(good)}")
print(f"Inliers:  {n_inlier}  ({100*n_inlier/len(good):.1f}%)")
print(f"Outliers: {n_outlier}")
print(f"\nHomography H  (img2 → img1 coordinate frame):")
print(np.array2string(H, precision=5, suppress_small=True))

# Visualize inliers (green) vs outliers (red)
inlier_mask = mask.ravel().tolist()
vis_ransac = cv2.drawMatches(img1, kps1, img2, kps2, good, None,
                              matchColor=(0, 255, 0),
                              singlePointColor=(200, 200, 200),
                              matchesMask=inlier_mask,
                              flags=cv2.DrawMatchesFlags_DEFAULT)
show(vis_ransac,
     f"Step 3 — RANSAC result: {n_inlier} inliers (green) / {n_outlier} outliers")


---
## Step 4 — Perspective Warp and Alpha Blending

### Computing the output canvas

We project the **four corners of image 2** through $H$ to find where they land in
image 1's coordinate frame. Together with image 1's own corners, this defines the
bounding box of the final panorama.

If any projected corner has a **negative coordinate**, we prepend a translation
matrix $T$ so that all pixels map to non-negative positions:

$$
H_{\text{shifted}} = T \cdot H, \quad
T = \begin{bmatrix} 1 & 0 & t_x \\ 0 & 1 & t_y \\ 0 & 0 & 1 \end{bmatrix}
$$

### Blending the overlap region

In the area covered by both images we apply a **simple 50/50 average**:

$$
I_{\text{out}}(p) = 0.5 \cdot I_1(p) + 0.5 \cdot I_2^{\text{warped}}(p)
$$

More sophisticated options (multi-band blending, Poisson blending) can reduce
visible seams but are beyond the scope of this demo.


In [ ]:
def warp_and_blend(img1: np.ndarray, img2: np.ndarray,
                   H: np.ndarray) -> np.ndarray:
    """
    Warp img2 into img1's coordinate frame using Homography H, then
    merge both images onto a common canvas.

    Steps:
      a) Project the four corners of img2 through H; compute canvas bounds.
      b) If corners land at negative coordinates, prepend a translation T.
      c) Apply warpPerspective to img2 with the shifted homography T @ H.
      d) Paste img1 onto the canvas at its offset position (tx, ty).
      e) In the overlap region, blend both images with equal 50/50 weights.

    Args:
        img1 : Reference (left) image, shape (H1, W1, 3).
        img2 : Query (right) image to be warped, shape (H2, W2, 3).
        H    : Homography mapping img2 coordinates -> img1 coordinates.

    Returns:
        panorama : Stitched output image.
    """
    h1, w1 = img1.shape[:2]
    h2, w2 = img2.shape[:2]

    # Project the four corners of img2 into img1's coordinate frame
    corners2 = np.float32([[0,0],[w2,0],[w2,h2],[0,h2]]).reshape(-1,1,2)
    warped_corners = cv2.perspectiveTransform(corners2, H)

    # Combine with img1's own corners to compute the full canvas extent
    corners1 = np.float32([[0,0],[w1,0],[w1,h1],[0,h1]]).reshape(-1,1,2)
    all_corners = np.concatenate([corners1, warped_corners], axis=0)

    x_min, y_min = np.floor(all_corners.min(axis=0).ravel()).astype(int)
    x_max, y_max = np.ceil (all_corners.max(axis=0).ravel()).astype(int)

    # Translation offset to shift all coordinates into the positive quadrant
    tx = max(-x_min, 0)
    ty = max(-y_min, 0)

    canvas_w = x_max - x_min
    canvas_h = y_max - y_min

    # Build shifted homography: apply translation on top of H
    T = np.array([[1,0,tx],[0,1,ty],[0,0,1]], dtype=np.float64)
    H_shifted = T @ H
    warped2 = cv2.warpPerspective(img2, H_shifted, (canvas_w, canvas_h))

    # Place img1 onto the canvas at its offset position
    canvas = warped2.copy()
    canvas[ty:ty+h1, tx:tx+w1] = img1

    # Identify the overlap region between img1's footprint and warped img2
    mask1 = np.zeros((canvas_h, canvas_w), dtype=np.uint8)
    mask1[ty:ty+h1, tx:tx+w1] = 255
    mask2   = (warped2.sum(axis=2) > 0).astype(np.uint8) * 255
    overlap = cv2.bitwise_and(mask1, mask2)

    # Blend overlap with equal weights (simple average)
    if overlap.any():
        region1 = canvas [overlap > 0].astype(np.float32)
        region2 = warped2[overlap > 0].astype(np.float32)
        canvas[overlap > 0] = (0.5*region1 + 0.5*region2).astype(np.uint8)

    return canvas

panorama = warp_and_blend(img1, img2, H)
print(f"Panorama size: {panorama.shape[1]}×{panorama.shape[0]}")

show(panorama, f"Step 4 — Stitched Panorama  ({panorama.shape[1]}×{panorama.shape[0]})")


---
## Step 5 — Final Summary and Results

Below we display all intermediate results side-by-side for a complete visual summary
of the pipeline.


In [ ]:
# ── Collect all visualization images ──────────────────────────────────────

# Keypoints
vis_kp1 = cv2.drawKeypoints(img1, kps1, None,
                             flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS)
vis_kp2 = cv2.drawKeypoints(img2, kps2, None,
                             flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS)

# All good matches (after ratio test, before RANSAC)
vis_all = cv2.drawMatches(img1, kps1, img2, kps2, good, None,
                           matchColor=(0,200,255),
                           singlePointColor=(200,200,200),
                           flags=cv2.DrawMatchesFlags_DEFAULT)

# Inlier matches only (after RANSAC)
vis_inliers = cv2.drawMatches(img1, kps1, img2, kps2, good, None,
                               matchColor=(0,255,0),
                               singlePointColor=(200,200,200),
                               matchesMask=mask.ravel().tolist(),
                               flags=cv2.DrawMatchesFlags_DEFAULT)

# ── Layout ─────────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(18, 14))
gs  = gridspec.GridSpec(3, 2, figure=fig, hspace=0.35, wspace=0.05)

ax = [fig.add_subplot(gs[0, 0]),
      fig.add_subplot(gs[0, 1]),
      fig.add_subplot(gs[1, 0]),
      fig.add_subplot(gs[1, 1]),
      fig.add_subplot(gs[2, :])]

show(vis_kp1,
     f"Step 1a — Image 1 SIFT keypoints ({len(kps1)})", ax[0])
show(vis_kp2,
     f"Step 1b — Image 2 SIFT keypoints ({len(kps2)})", ax[1])
show(vis_all,
     f"Step 2 — FLANN matches after ratio test ({len(good)})", ax[2])
show(vis_inliers,
     f"Step 3 — RANSAC inliers ({n_inlier}) / outliers ({n_outlier})", ax[3])
show(panorama,
     f"Step 4 — Final Panorama  ({panorama.shape[1]}×{panorama.shape[0]})", ax[4])

fig.suptitle("Image Stitching Pipeline — Full Summary", fontsize=16, fontweight="bold")
plt.show()

# ── Print Homography ────────────────────────────────────────────────────────
print("=" * 55)
print("  Estimated Homography H  (img2 → img1 frame)")
print("=" * 55)
print(np.array2string(H, precision=5, suppress_small=True))
print()
print(f"  Good matches (ratio test) : {len(good)}")
print(f"  RANSAC inliers            : {n_inlier}  ({100*n_inlier/len(good):.1f}%)")
print(f"  Panorama size             : {panorama.shape[1]}×{panorama.shape[0]}")

# ── Save outputs ────────────────────────────────────────────────────────────
cv2.imwrite("panorama.jpg",      panorama)
cv2.imwrite("matches_inlier.jpg", vis_inliers)
print()
print("Saved: panorama.jpg, matches_inlier.jpg")
